# AgenticML — Experimento 2: Titanic (Classificação Binária)

**Dataset:** Titanic (seaborn)  
**Tipo de problema:** Classificação binária (sobreviveu / não sobreviveu)  
**Objetivo:** Prever se um passageiro sobreviveu ao naufrágio do Titanic  
**Referência de comparação:** Auto-sklearn (seção 9.1 da spec)  

---
**Relevância:** Este experimento é mais rico que o Iris pois exercita:
- Agente 2: detecção de missing values (coluna `age`) e desbalanceamento de classes
- Agente 3: imputação, One-Hot Encoding em variáveis nominais (`sex`, `embarked`, `class`)
- Agente 5: comparação de modelos em dataset com ruído real

In [1]:
import sys, os, time, warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import seaborn as sns

from core.graph import build_graph
from core.state import initial_state
from tcc_eval import make_split, evaluate_agenticml


## 1. Dataset — Titanic

| Atributo | Valor |
|----------|-------|
| Instâncias | ~891 |
| Features selecionadas | pclass, sex, age, sibsp, parch, fare, embarked |
| Target | survived (0 = não sobreviveu, 1 = sobreviveu) |
| Missing values | age (~20%), embarked (2 linhas) |
| Desbalanceamento | ~62% não sobreviveram / ~38% sobreviveram |
| Fonte | `seaborn.load_dataset('titanic')` |

In [2]:
titanic_raw = sns.load_dataset('titanic')

# Seleciona colunas relevantes (remove duplicatas/leakage)
COLS = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'survived']
df = titanic_raw[COLS].copy()

print('Shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\nDistribuição da target:')
print(df['survived'].value_counts())
print(f'  → Classe majoritária: {df["survived"].value_counts(normalize=True).max()*100:.1f}%')
display(df.head())

Shape: (891, 8)

Missing values:
age         177
embarked      2
dtype: int64

Distribuição da target:
survived
0    549
1    342
Name: count, dtype: int64
  → Classe majoritária: 61.6%


,pclass,sex,age,sibsp,parch,fare,embarked,survived
0,3,male,22.0,1,0,7.2500,S,0
1,1,female,38.0,1,0,71.2833,C,1
2,3,female,26.0,0,0,7.9250,S,1
3,1,female,35.0,1,0,53.1000,S,1
4,3,male,35.0,0,0,8.0500,S,0


In [3]:
csv_bytes = df.to_csv(index=False).encode('utf-8')
print(f'CSV gerado: {len(csv_bytes):,} bytes')

CSV gerado: 23,182 bytes


## 2. Execução do Pipeline AgenticML

In [4]:
PROMPT = (
    "Prever se um passageiro sobreviveu ao naufrágio do Titanic "
    "com base nas características socioeconômicas e demográficas."
)

# Protocolo de avaliação (parecer da orientação): holdout isolado da seleção de modelos.
SEED = 42
train_df, test_df, split_info = make_split(
    df, target='survived', task='classification', test_size=0.2, seed=SEED
)
print(f"Split: {split_info['n_train']} treino / {split_info['n_test']} teste "
      f"(estratificado={split_info['stratified']}, seed={SEED})")

t0 = time.time()
ev = evaluate_agenticml(
    train_df, test_df, target='survived', prompt=PROMPT,
    hint='classification', seed=SEED, return_state=True,
)
elapsed = time.time() - t0
result = ev['_final_state']

print(f'\n✓ Pipeline concluído em {elapsed:.2f}s')
print('Status final:', result.get('status'))


Split: 712 treino / 179 teste (estratificado=True, seed=42)

✓ Pipeline concluído em 24.56s
Status final: done


## 3. Resultados por Agente

### Agente 1 — Ingestão e Inspeção

In [5]:
inspection = result.get('inspection', {})
schema = inspection.get('schema', {})

print('Shape detectado:', schema.get('shape'))
print('Encoding:', inspection.get('encoding_detected'))
print()
stats = inspection.get('descriptive_stats', {})
print('Colunas com nulls detectados pelo Agente 1:')
for col, s in stats.items():
    nulls = s.get('nulls', 0)
    if nulls > 0:
        print(f'  {col}: {nulls} nulos')

Shape detectado: [712, 8]
Encoding: utf-8

Colunas com nulls detectados pelo Agente 1:
  age: 137 nulos
  embarked: 2 nulos


### Agente 2 — Diagnóstico de Qualidade

In [6]:
quality = result.get('quality', {})
issues = quality.get('issues', [])

print(f"Overall severity: {quality.get('overall_severity')}")
print(f"Issues detectados: {len(issues)}\n")

for i in issues:
    print(f"[{i['severity'].upper()}] {i['type']}")
    print(f"  → {i['alert']}\n")

Overall severity: medium
Issues detectados: 5

[MEDIUM] missing_values
  → A coluna 'age' possui 137 valores ausentes, o que representa 19.24% do total. Recomenda-se tratar esses valores ausentes, possivelmente utilizando a imputação com a média ou mediana, ou categorizando a idade em faixas etárias.

[LOW] missing_values
  → A coluna 'embarked' possui 2 valores ausentes, representando 0.28% do total. Recomenda-se preencher esses valores ausentes com a moda, que é 'S', ou considerar a exclusão dessas linhas se o impacto for mínimo.

[LOW] class_imbalance
  → A coluna 'sex' apresenta um desequilíbrio de classes, com 64.47% dos registros sendo do sexo masculino. Considere técnicas de balanceamento, como reamostragem, para evitar viés no modelo.

[LOW] class_imbalance
  → A coluna 'embarked' apresenta um desequilíbrio de classes, com 72.68% dos registros embarcando no porto 'S'. Considere técnicas de balanceamento para garantir que o modelo não seja tendencioso em relação ao porto de emba

### Agente 3 — Feature Engineering

In [7]:
features = result.get('features', {})
transforms = features.get('transformations_applied', [])

print(f"Target:   {features.get('target_column')}")
print(f"Features: {features.get('feature_columns')}")
print()
for t in transforms:
    print(f"• {t['type']} em {t['columns']}")
    print(f"  → {t['justification']}")

Target:   survived
Features: ['pclass', 'age', 'sibsp', 'parch', 'fare', 'sex_male', 'embarked_Q', 'embarked_S']

• impute_median em ['age']
  → A coluna 'age' possui valores ausentes e a mediana  e9 uma boa escolha para imputar dados num e9ricos, pois  e9 robusta a outliers.
• impute_most_frequent em ['embarked']
  → A coluna 'embarked' possui poucos valores ausentes e  e9 categ f3rica, portanto, a imputa e7 e3o pela moda (valor mais frequente)  e9 adequada.
• one_hot_encoding em ['sex']
  → A coluna 'sex'  e9 nominal com duas categorias, portanto, o one-hot encoding  e9 apropriado.
• one_hot_encoding em ['embarked']
  → A coluna 'embarked'  e9 nominal com tr eas categorias, portanto, o one-hot encoding  e9 apropriado.
• label_encoding em ['pclass']
  → A coluna 'pclass'  e9 ordinal, pois representa classes sociais, portanto, o label encoding  e9 apropriado.
• standard_scaler em ['age', 'fare']
  → As colunas 'age', 'fare' e 'family_size' s e3o num e9ricas e n e3o possuem limites natu

### Agente 4 — Interpretação de Linguagem Natural

In [8]:
interp = result.get('interpretation', {})
print(f"Tipo:       {interp.get('problem_type')}")
print(f"Confiança:  {interp.get('confidence')}")
print(f"Divergiu:   {interp.get('diverged_from_user')}")
print(f"\nJustificativa: {interp.get('justification')}")

Tipo:       classification
Confiança:  high
Divergiu:   False

Justificativa: O objetivo do usuário é prever se um passageiro sobreviveu ao naufrágio do Titanic, o que é uma tarefa de classificação, pois envolve prever uma categoria discreta: 'sobreviveu' ou 'não sobreviveu'. A coluna target identificada, 'survived', é binária, indicando se o passageiro sobreviveu (1) ou não (0). As características do dataset, como 'pclass', 'age', 'sibsp', 'parch', 'fare', 'sex_male', 'embarked_Q', e 'embarked_S', são usadas para prever essa categoria. Portanto, o tipo de problema é inequivocamente de classificação.


### Agente 5 — Seleção e Treinamento

In [9]:
training = result.get('training', {})

print(f"Modelo selecionado: {training.get('model_selected')}")
print(f"F1-score (weighted, CV/seleção): {training.get('metric_value')}")
print()

candidates = training.get('candidate_results', [])
df_cands = pd.DataFrame(candidates).sort_values('score', ascending=False)
display(df_cands)

print(f"\nJustificativa: {training.get('justification')}")

print('\n=== Validação honesta (holdout isolado, nunca visto na seleção) ===')
print(f"F1-score no holdout: {ev['holdout_metric']}")
print(f"Gap de otimismo (CV − holdout): {ev['optimism_gap']}")
if ev['optimism_gap'] is not None and abs(ev['optimism_gap']) > 0.02:
    print('  ⚠ Gap > 0.02 — reportar o valor de holdout no TCC, não o de CV.')


Modelo selecionado: XGBoost
F1-score (weighted, CV/seleção): 0.8364



,model,score
2,XGBoost,0.8364
3,SVM,0.8210
1,Random Forest,0.8194
0,Logistic Regression,0.7945



Justificativa: XGBoost foi selecionado com F1-score ponderado de 0.8364 para o problema de classification. Demais candidatos avaliados: SVM (0.8210), Random Forest (0.8194), Logistic Regression (0.7945). Hiperparâmetros otimizados via busca aleatória (CV): {'learning_rate': 0.2922, 'max_depth': 3, 'n_estimators': 121, 'subsample': 0.6028}.

=== Validação honesta (holdout isolado, nunca visto na seleção) ===
F1-score no holdout: 0.803
Gap de otimismo (CV − holdout): 0.0334
  ⚠ Gap > 0.02 — reportar o valor de holdout no TCC, não o de CV.


### Agente 6 — Previsão e Explicação

In [10]:
output = result.get('output', {})
explanation = output.get('explanation', {})
importances = output.get('feature_importances', [])

print('=== Explicação Técnica ===')
print(explanation.get('technical', ''))

print('\n=== Explicação Acessível ===')
print(explanation.get('accessible', ''))

if importances:
    print('\n=== Importância das Features ===')
    display(pd.DataFrame(importances))

=== Explicação Técnica ===
O pipeline AgenticML foi aplicado a um dataset com 712 linhas e 8 colunas, focando na previsão de sobrevivência de passageiros do Titanic. O Agente 2 identificou problemas de qualidade, como valores ausentes e duplicados, com severidade geral média. O Agente 3 realizou transformações de engenharia de features, incluindo imputação de mediana para 'age', imputação do valor mais frequente para 'embarked', codificação one-hot para 'sex' e 'embarked', e codificação de rótulos para 'pclass'. Além disso, aplicou-se a normalização com Standard Scaler em 'age' e 'fare'.

O Agente 4 confirmou que o problema é de classificação, com a variável alvo 'survived' sendo binária. O Agente 5 selecionou o modelo XGBoost, que apresentou o melhor desempenho com um F1-score ponderado de 0.8364, superando outros modelos como SVM (0.8210), Random Forest (0.8194) e Logistic Regression (0.7945). A seleção do XGBoost foi justificada pela sua capacidade de lidar com o problema de classif

,feature,importance
0,sex_male,0.378079
1,pclass,0.178672
2,embarked_S,0.090713
3,sibsp,0.082783
4,fare,0.081201
5,age,0.077563
6,embarked_Q,0.055880
7,parch,0.055109


## 4. Resumo — Tabela para TCC

In [11]:
schema = result.get('inspection', {}).get('schema', {})
issues = result.get('quality', {}).get('issues', [])
transforms = result.get('features', {}).get('transformations_applied', [])

summary = {
    'Experimento':             'Titanic — Classificação Binária',
    'Dataset':                 f"{schema.get('shape', [0,0])[0]} linhas × {schema.get('shape', [0,0])[1]} colunas",
    'Split':                   f"{split_info['n_train']} treino / {split_info['n_test']} teste (seed={SEED})",
    'Tipo de problema':        interp.get('problem_type', 'N/A'),
    'Issues de qualidade':     len(issues),
    'Transformações':          len(transforms),
    'Modelo selecionado':      training.get('model_selected', 'N/A'),
    'Métrica':                 'f1_weighted',
    'Score CV (seleção)':      training.get('metric_value', 'N/A'),
    'Score holdout (honesto)': ev['holdout_metric'],
    'Gap de otimismo':         ev['optimism_gap'],
    'Tempo (s)':               round(elapsed, 2),
}

display(pd.DataFrame([summary]).T.rename(columns={0: 'Valor'}))


,Valor
Experimento,Titanic — Classificação Binária
Dataset,712 linhas × 8 colunas
Split,712 treino / 179 teste (seed=42)
Tipo de problema,classification
Issues de qualidade,5
Transformações,6
Modelo selecionado,XGBoost
Métrica,f1_weighted
Score CV (seleção),0.8364
Score holdout (honesto),0.803
